In [1]:
import pandas as pd
from transformers import T5Tokenizer,Trainer, TrainingArguments, T5ForConditionalGeneration

/Users/ravihw18/Projects/Test Summerizer/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_data=pd.read_csv("/Users/ravihw18/Projects/Test Summerizer/Dataset/samsum-train.csv")
val_data=pd.read_csv("/Users/ravihw18/Projects/Test Summerizer/Dataset/samsum-validation.csv")

In [3]:
train_data.shape

(14732, 3)

In [4]:
val_data.shape

(818, 3)

In [5]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [6]:
train_data.shape

(4000, 3)

# Data Pre-Processing

In [7]:
import re
def clean_data(text):
    text=re.sub(r"\r\n"," ", text) #Lines
    text=re.sub(r"\s+"," ", text) #Extra spaces
    text=re.sub(r"<.*?>"," ", text) #Remove html tags <p> <h1>
    text=text.strip().lower() #Remove leading and trailing spaces and lower() for converting to lower case
    return text

In [8]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

# Tokenization

In [9]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [10]:
# raw data -> tokenized inputs for finetuning 

def tokenize(data):
    inputs=tokenizer(data['dialogue'], padding="max_length", truncation=True, max_length=512)
    targets=tokenizer(data['summary'], padding="max_length", truncation=True, max_length=150)

    inputs['labels']=targets['input_ids'] 
    return inputs

In [11]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

# Working with our Model

In [12]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 15060.96it/s]


In [13]:
import torch
if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

print("device:", device)
model.to(device)

device: mps


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [14]:
# Training Arguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500,

)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
# Train the model
trainer.train()

/Users/ravihw18/Projects/Test Summerizer/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.629413,0.381781
2,0.396572,0.358890


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 10.57it/s]
/Users/ravihw18/Projects/Test Summerizer/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 10.05it/s]
/Users/ravihw18/Projects/Test Summerizer/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
